## GTSEP v1 multi

### Description
Same model as v1 but with multiple years (longer planning horizon). This means that batteries are unconstrained in investments.

The model does not include degradation of assets, so all assets have an infinite lifetime and new investments are just added to existing ones. The key difference here with regard to implementation is two things:

1. Since the model is over multiple years, the objective function needs to consider the time value of money. I have solved this by calulcating the annualized investment costs to represent the capex costs and the opex costs are simply the cost of production. Both are discounted to the present value using the discount rate.
2. Since the model is over multiple years, basically all decision variables as well as input data has been extended with the year index y. I use the convention that for timesteps the year index comes first, then the timestep index (hour).

### Indexes and index sets

- $n \in N$: Set of nodes.
- $i \in G^{old}$: Set of existing generators. Each generator is associated with a node $n$.
- $i \in G^{new}$: Set of new generators. Each generator is associated with a node $n$.
- $i \in G$: Set of all generators.
- $i \in G_n$: Set of generators at node $n$, including new generators.
- $b \in B^{new}$: Set of new branches.
- $b \in B^{old}$: Set of existing branches.
- $b \in B$: Set all of branches.
- $b \in B_n^{in}$: Set of branches coming into node $n$, including new branches.
- $b \in B_n^{out}$: Set of branches going out of node $n$, including new branches.
- $s \in S^{old}$: Set of new batteries. Each battery is associated with a node $n$.
- $s \in S^{new}$: Set of new batteries. Each battery is associated with a node $n$.
- $s \in S$: Set of all batteries. Each battery is associated with a node $n$.
- $s \in S_n$: Set of batteries at node $n$, including new batteries.
- $t \in T$: Set of time periods.
- $y \in Y$: Set of years.
- $ y' \in Y_{y} $: Set of years up to year $y$. $Y_y = \{y' \in Y | y' \leq y \}$  i.e. if $Y = [2025, 2035, 2045]$ and $y = 2035$, then $Y_{y} = [2025, 2035]$.

### Parameters

- $P_{i}^{\min}$: Minimum power output of generator $i$ (MW)
- $P_{i}^{\max}$: Maximum power output of generator $i$ (MW)
- ${VOLL_y}$: Value of lost load (cost of load shedding) ($/MWh)
- $CC_y$: Cost of curtailment (\$/MWh)
- $MC_{i}$: Marginal cost of generator $i$ (\$/MWh)
- $CO2_{i,y}$ Cost of CO2 emissions of generator $i$ (\$/MWh)
- $E_{i}$: CO2 emissions of generator $i$ (ton/MWh)
- $E_{limit}$: CO2 emissions limit (ton)
- $D_{n,y,t}$: Demand at node $n$ and time $t$ (MW)
- $l_b$: loss factor of branch $b$ (given, but in reality some function of distance, transmsission line type, etc.)
- $P_{b,y}^{\max}$: Maximum power flow on branch $b$ (MW)
- $\eta_{s}^{ch}$: charge efficiency of battery s
- $\eta_{s}^{dis}$: discharge efficiency of battery s
- $P_{s}^{ch, \max}$: Maximum charging power of battery $s$ (MW)
- $P_{s}^{ch, \min}$: Minimum charging power of battery $s$ (MW)
- $P_{s}^{dis, \max}$: Maximum discharge power output of battery $s$ (MW)
- $P_{s}^{dis, \min}$: Minimum discharge power output battery $s$ (MW)
- $SOC_{s}^{\max}$: Maximum state of charge of battery $s$ (MWh)
- $SOC_{s}^{\min}$: Minimum state of charge of battery $s$ (MWh)
- $MC_{s,y}^{dis}$: Marginal cost of discharging battery $s$ (\$/MWh)
- $cf_{i,y, t}$: Capacity factor of generator $i$ at time $t$. Equals 1 for all non-renewable generators, otherwise $\in (0, 1)$.
- $P_b^{min}$: Minimum capacity of new branch $b$ (MW)
- $P_{b}^{\max}$: Maximum power flow on branch $b$ (MW)
- $AIC_{i,y}$: Annualized investment cost of new generator $i$ if made in year $y$ (\$/MW)
- $AIC_{s,y}$: Annualized investment cost of new battery $s$ if made in year $y$ (\$/MWh)
- $AIC_{b,y}$: Annualized investment cost of new branch $b$ if made in year $y$ (\$/MW)

### Decision variables

- $g_{i,y,t}$: Power generation dispatch of generator $i$ in year $y$ at time $t$ (MW)
- $f_{b,y,t}$: Power flow on branch $b$ in year $y$ at time $t$ (MW)
- $sh_{n,y,t}$: Load shedding at node $n$ in year $y$ at time $t$ (MW)
- $c_{i,y,t}$: Power curtailment at generator $i$ in year $y$ at time $t$ (MW)
- $g^{ch}_{s,y,t}$: Charging power of battery $s$ in year $y$ at time $t$ (MW)
- $g^{dis}_{s,y,t}$: Discharging power of battery $s$ in year $y$ at time $t$ (MW)
- $soc_{s,y,t}$: State of charge of storage unit $s$ in year $y$ at time $t$ (MWh)
- $soc_{s,y}^{max}$: Energy capacity built of storage unit $s$ in year $y$ (MWh)
- $p_{i,y}^{max}$: The capacity built of new generator i in year $y$ (MW)
- $p_{b,y}^{max}$: Maximum capacity of new branch b in year $y$ (MW)

### Optimization Model

### Objective function, minimize cost of generation

**Minimize:**
$$
\frac{1}{|Y|} \sum_{y \in Y} \frac{1}{(1+r)^{y-Y[0]}} \left( AIC_y + OC_y \right)
$$

where

$$
OC_y = \sum_{i \in G} \sum_{t \in T} \underbrace{(MC_{i,y} + CO2_{i,y}) g_{i,y,t}}_{\text{Generation cost}} 
+ \sum_{s \in S} \sum_{t \in  T} \underbrace{MC_{s,y} g_{s,y,t}^{dis} \eta_{s}^{dis}}_{\text{Battery discharge cost}}
+ \sum_{n \in N} \sum_{t \in T} \underbrace{sh_{n,y,t} VOLL_y}_{\text{Load shedding cost}} 
+ \sum_{n \in N} \sum_{t \in T} \underbrace{c_{n,y,t} CC_y}_{\text{Curtailment cost}} 
$$

and

$$ 
AIC_y = \underbrace{\sum_{i \in G^{new}} AIC_{i,y} p_{i,y}^{max}}_{\text{Generator expansion cost}} 
+ \underbrace{\sum_{b \in B^{new}} AIC_{b,y} p_{b,y}^{max}}_{\text{Transmission expansion cost}} 
+ \underbrace{\sum_{s \in S^{new}} AIC_{s,y} soc_{s,y}^{max}}_{\text{Storage expansion cost}} 
$$


1. **Power balance: production + inflow - curtailment = outflow + demand - shedding**

_A.K.A. Market clearing or energy balance_

$$ \sum_{i \in G_n} (g_{i,y,t} - c_{i,y,t}) + \sum_{b \in B_n^{in}} f_{b,y,t} (1 - l_{b}) - \sum_{b \in B_n^{out}} f_{b,y,t} - \sum_{s \in S_n} (g_{s,y,t}^{ch} - \eta_{s}^{dis} g_{s,y,t}^{dis}) + sh_{n,y,t} = D_{n,y,t} \quad \forall n \in N, \forall y \in Y, \forall t \in T $$

2. **a) We can't shed more load than the demand**

$$ sh_{n,y,t} \leq D_{n,y,t} \quad \forall n \in N, \forall y \in Y, \forall t \in T $$

2. **b) We can't curtail more energy than is produced**

$$ 0 \leq c_{i,y,t} \leq g_{i,y,t} \quad \forall i \in G, \forall y \in Y, \forall t \in T $$

3. **a) Generators' power output limits, old generators**

$$ P_{i}^{\min} \leq g_{i,y,t} \leq P_{i}^{\max} {cf}_{i,y,t} \quad \forall i \in G^{old}, \forall y \in Y, \forall t \in T $$

where ${cf}_{i,y,t}$ is 1 $ \forall t$ if the generator is fossil, but varies if the generator is renewable.

3. **b) Generators' power output limits, new generators**

$$ 0 \leq g_{i,y,t} \leq cf_{i,y,t} \sum_{y'\in Y_y} \left( p_{i,y'}^{\max} \right)  \quad \forall i \in G^{new}, \forall y \in Y, \forall t \in T $$

3. **c) New generators' installed capacity limit**

$$ p_{i,y}^{max} \leq P_i^{\max} \quad \forall i \in G^{new}, \forall y \in Y $$

4. **a) Branch power flow limits, old branches**

$$ -P_{b}^{\max} \leq f_{b,y,t} \leq P_{b}^{\max} \quad \forall b \in B^{old}, \forall y \in Y, \forall t \in T $$

4. **b) Branch power flow limits, new branches**

$$ -\sum_{y' \in Y_y} p_{b,y'}^{\max} \leq f_{b,y,t} \leq \sum_{y' \in Y_y} p_{b,y'}^{\max} \quad \forall b \in B^{new}, \forall y \in Y, \forall t \in T $$

4. **c) Branch max_capacity limits, new branches**

$$  p_{b,y}^{\max} \leq P_{b}^{\max} \quad \forall b \in B^{new}, \forall y \in Y $$

5. **Emissions restrictions**

$$ \sum_{i \in G} \sum_{y \in Y} \sum_{t \in T} E_{i,y} g_{i,y,t} \leq E_{limit} $$

6. **a) Battery charging limit, old batteries**

$$ P_{s}^{ch, \min} \leq g_{s,y,t}^{ch} \leq P_{s}^{ch, \max} \quad \forall s \in S^{old}, \forall y \in Y, \forall t \in T $$

6. **b) Battery charging limit, new batteries**

$$ 0 \leq g_{s,y,t}^{ch} \leq \frac{\sum_{y' \in Y_y} soc_{s,y'}^{\max}}{batt_{hours} \cdot {cdrate}} \quad \forall s \in S^{new}, \forall y \in Y, \forall t \in T $$

7. **a) Battery discharging limit, old batteries**

$$ P_{s}^{dis, \min} \leq g_{s,y,t}^{dis} \leq P_{s}^{dis, \max} \quad \forall s \in S^{old}, \forall y \in Y, \forall t \in T $$

7. **b) Battery discharging limit, new batteries**

$$ 0 \leq g_{s,y,t}^{dis} \leq \frac{\sum_{y' \in Y_y} soc_{s,y'}^{\max}}{batt_{hours}} \quad \forall s \in S^{new}, \forall y \in Y, \forall t \in T $$

8. **Battery state of charge limits**

$$ SOC_{s}^{\min} \cdot soc_{s,y}^{max} \leq soc_{s,y,t} \leq SOC_{s}^{\max}\cdot soc_{s,y}^{max} \quad \forall s \in S, \forall y \in Y, \forall t \in T $$

9. **Battery state of charge dynamics**

$$ soc_{s,y,t} = soc_{s,y,t-1} + \eta_{s}^{ch} g_{s,y,t}^{ch} - \frac{1}{\eta_{s}^{dis}} g_{s,y,t}^{dis} \quad \forall s \in S, \forall y \in Y, \forall t \in T - \{0\} $$

10. **a) Battery state of charge at time 0**

$$ soc_{s,y,0} = SOC_{s}^{\min} \cdot soc_{s,y}^{max} \quad \forall s \in S, \forall y \in Y $$

10. **b) Battery state of charge at time T[-1]**

$$ soc_{s,y,T[-1]} = SOC_{s}^{\min} \cdot soc_{s,y}^{max} \quad \forall s \in S, \forall y \in Y $$

11. **Variable definitions**

All continuous variables are non-negative:
$$ 
g_{i,y,t}, f_{b,y,t}, sh_{n,y,t}, c_{n,y,t}, g_{s,y,t}^{ch}, g_{s,y,t}^{dis}, soc_{s,y,t}, p_{i,y}^{max}, p_{b,y}^{max}, soc_{s,y}^{\max} \geq 0, 
\quad \forall i \in G, \forall b \in B, \forall n \in N, \forall s \in S, \forall t \in T 
$$

All binary variables are restricted to 0 or 1:
$$ 
xi_{i,y}, xb_{b,y}, xs_{s,y} \in \{0,1\}, 
\quad \forall i \in G^{new}, \forall b \in B^{new}, \forall s \in S^{new},  \forall y \in Y
$$